In [1]:
import os
import random
import shutil

# Đường dẫn gốc (Input) và đích (Working)
SRC_BASE = '/kaggle/input/datasets/pandapowr/labeled-trash-dataset-pascal-voc-coco-and-yolo/splittedAugmentatedData/splittedAugmentatedData/yolo'
DST_BASE = '/kaggle/working/dataset'

# Tỉ lệ lấy dữ liệu
SAMPLE_RATIO = 0.5 

splits = ['train', 'val', 'test']

# Cố định random seed để mỗi lần chạy lại lấy đúng 50% data giống nhau
random.seed(42)

# Xử lý từng tập train, val, test
for split in splits:
    # Đường dẫn nguồn theo đúng cấu trúc ảnh bạn gửi
    src_images_dir = os.path.join(SRC_BASE, split, 'images')
    src_labels_dir = os.path.join(SRC_BASE, split, 'labels')
    
    # Đường dẫn đích ở working
    dst_images_dir = os.path.join(DST_BASE, split, 'images')
    dst_labels_dir = os.path.join(DST_BASE, split, 'labels')
    
    # Tạo thư mục đích
    os.makedirs(dst_images_dir, exist_ok=True)
    os.makedirs(dst_labels_dir, exist_ok=True)
    
    if not os.path.exists(src_images_dir):
        print(f"Bỏ qua tập {split} vì không tìm thấy thư mục: {src_images_dir}")
        continue
        
    # Lấy danh sách ảnh
    image_files = [f for f in os.listdir(src_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    # Tính số lượng 50% và chọn ngẫu nhiên
    num_samples = int(len(image_files) * SAMPLE_RATIO)
    sampled_images = random.sample(image_files, num_samples)
    
    print(f"[{split.upper()}] Đang xử lý {num_samples}/{len(image_files)} file (gộp nhãn về 0)...")
    
    for img_name in sampled_images:
        # Copy ảnh
        src_img_path = os.path.join(src_images_dir, img_name)
        dst_img_path = os.path.join(dst_images_dir, img_name)
        shutil.copy(src_img_path, dst_img_path)
        
        # Xử lý label: chuyển class ID thành 0
        label_name = os.path.splitext(img_name)[0] + '.txt'
        src_label_path = os.path.join(src_labels_dir, label_name)
        dst_label_path = os.path.join(dst_labels_dir, label_name)
        
        if os.path.exists(src_label_path):
            with open(src_label_path, 'r') as f_in, open(dst_label_path, 'w') as f_out:
                for line in f_in:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        # Ghi lại với ID class = 0, giữ nguyên bounding box
                        parts[0] = '0'
                        f_out.write(' '.join(parts) + '\n')



[TRAIN] Đang xử lý 1723/3446 file (gộp nhãn về 0)...
[VAL] Đang xử lý 492/985 file (gộp nhãn về 0)...
[TEST] Đang xử lý 246/492 file (gộp nhãn về 0)...


In [2]:
import os
import json
import random
import shutil

# Đường dẫn TACO (Input) và đích (Working - nơi chứa data cũ)
TACO_BASE = '/kaggle/input/datasets/kneroma/tacotrashdataset/data'
JSON_PATH = os.path.join(TACO_BASE, 'annotations.json')
DST_BASE = '/kaggle/working/dataset'

# Đọc file COCO annotations
print("Đang đọc file annotations.json...")
with open(JSON_PATH, 'r') as f:
    coco_data = json.load(f)

images = coco_data['images']
annotations = coco_data['annotations']

# Gom nhóm các box theo từng ID ảnh
img_to_anns = {img['id']: [] for img in images}
for ann in annotations:
    # Bỏ qua các box bị lỗi không có tọa độ
    if 'bbox' in ann and len(ann['bbox']) == 4:
        img_to_anns[ann['image_id']].append(ann)

# 1. Lấy ngẫu nhiên 50% dữ liệu TACO
random.seed(42)
sample_size = int(len(images) * 0.5)
sampled_images = random.sample(images, sample_size)

# 2. Chia thành train/val/test (Tỉ lệ 80/10/10)
random.shuffle(sampled_images)
train_split = int(0.8 * len(sampled_images))
val_split = int(0.9 * len(sampled_images))

splits = {
    'train': sampled_images[:train_split],
    'val': sampled_images[train_split:val_split],
    'test': sampled_images[val_split:]
}

# Hàm chuyển đổi COCO format sang YOLO format
def coco_to_yolo(bbox, img_w, img_h):
    x_min, y_min, w, h = bbox
    
    # Tính tọa độ tâm và chuẩn hóa về 0-1
    x_center = (x_min + w / 2) / img_w
    y_center = (y_min + h / 2) / img_h
    norm_w = w / img_w
    norm_h = h / img_h
    
    # Ép giá trị không vượt quá giới hạn ảnh (tránh lỗi YOLO)
    x_center = max(0.0, min(1.0, x_center))
    y_center = max(0.0, min(1.0, y_center))
    norm_w = max(0.0, min(1.0, norm_w))
    norm_h = max(0.0, min(1.0, norm_h))
    
    return x_center, y_center, norm_w, norm_h

# 3. Xử lý ảnh và Labels
for split_name, imgs in splits.items():
    dst_img_dir = os.path.join(DST_BASE, split_name, 'images')
    dst_lbl_dir = os.path.join(DST_BASE, split_name, 'labels')
    
    print(f"[{split_name.upper()}] Đang xử lý và trộn {len(imgs)} ảnh từ TACO...")
    
    for img in imgs:
        # Tên file gốc (ví dụ: 'batch_1/000001.jpg')
        original_filename = img['file_name']
        src_img_path = os.path.join(TACO_BASE, original_filename)
        
        # Đổi tên file để tránh trùng lặp giữa các batch (thành 'batch_1_000001.jpg')
        safe_filename = original_filename.replace('/', '_').replace('\\', '_')
        dst_img_path = os.path.join(dst_img_dir, safe_filename)
        
        # Bỏ qua nếu không tìm thấy ảnh thực tế
        if not os.path.exists(src_img_path):
            continue
            
        # Copy ảnh
        shutil.copy(src_img_path, dst_img_path)
        
        # Tạo file label .txt
        label_filename = os.path.splitext(safe_filename)[0] + '.txt'
        dst_lbl_path = os.path.join(dst_lbl_dir, label_filename)
        
        img_w = img['width']
        img_h = img['height']
        anns = img_to_anns[img['id']]
        
        with open(dst_lbl_path, 'w') as f_out:
            for ann in anns:
                bbox = ann['bbox']
                # Nếu kích thước box = 0 thì bỏ qua
                if bbox[2] <= 0 or bbox[3] <= 0:
                    continue
                    
                x_c, y_c, w, h = coco_to_yolo(bbox, img_w, img_h)
                # Ghi lại với ID class = 0 (garbage)
                f_out.write(f"0 {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

print("\n[HOÀN TẤT] 50% dữ liệu TACO đã được trộn thành công vào /kaggle/working/dataset!")

Đang đọc file annotations.json...
[TRAIN] Đang xử lý và trộn 600 ảnh từ TACO...
[VAL] Đang xử lý và trộn 75 ảnh từ TACO...
[TEST] Đang xử lý và trộn 75 ảnh từ TACO...

[HOÀN TẤT] 50% dữ liệu TACO đã được trộn thành công vào /kaggle/working/dataset!


In [3]:
dataset_path = '/kaggle/working/dataset'
yaml_path = os.path.join(dataset_path, 'data.yaml')

yaml_content = f"""path: {dataset_path}
train: train/images
val: val/images
test: test/images

nc: 1
names: ['garbage']
"""

with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print(f"✅ Đã tạo thành công file cấu hình tại: {yaml_path}")

✅ Đã tạo thành công file cấu hình tại: /kaggle/working/dataset/data.yaml


In [4]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.0 MB/s eta 0:00:0000:01


In [5]:
import os
from ultralytics import YOLO
model = YOLO('/kaggle/input/models/trmynguyen/garbage-detect-v2/pytorch/default/1/best.pt') 

print("Bắt đầu quá trình training...")

results = model.train(
    data=yaml_path,         
    epochs=250,                 # tăng số epoch
    imgsz=896,                   # ảnh lớn hơn giúp detect vật nhỏ tốt hơn
    batch=16,                    # giữ nguyên nếu VRAM đủ
    device=[0, 1],
    project='/kaggle/working/runs',
    name='garbage_detector_v2',
    patience=60,                 # early stopping chậm hơn
    workers=4,

    # Optimizer
    optimizer='AdamW',           # thường fine-tune tốt hơn SGD
    lr0=1e-3,                    # learning rate khởi đầu
    lrf=1e-2,                    # learning rate cuối = lr0 * lrf
    weight_decay=5e-4,

    # Warmup
    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    # Regularization
    dropout=0.1,                 # nếu model hỗ trợ
    close_mosaic=10,             # tắt mosaic ở 10 epoch cuối

    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,

    # Training tricks
    cos_lr=True,                 # cosine learning rate schedule
    amp=True,                    # mixed precision
    cache=True,                  # cache dataset vào RAM
    pretrained=True,              
    val=True,
    plots=True           
)

print("🎉 Training hoàn tất!")
print("Trọng số (weights) tốt nhất của bạn đã được lưu tại: /kaggle/working/runs/garbage_detector/weights/best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Bắt đầu quá trình training...
Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=10.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=250, erasing=0.4, exist_ok=False, fli